| TESTE BERT BASE |
Cálculo SPAN com 3 épocas e 30 experimentos no VSCode.

In [1]:
# Se não ativar automaticamente, ativar a .venv com .\.venv\Scripts\activate

In [2]:
# Instalação de tudo o que é necessário para rodar o projeto com pip install -r requirements.txt

In [3]:
# Importações necessárias

import pandas as pd
import csv
import re
import torch
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Optional
from datasets import Dataset, DatasetDict
from transformers import (AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification, EarlyStoppingCallback)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from datasets import concatenate_datasets

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

import evaluate
metric = evaluate.load("seqeval")

c:\Users\Myllena\Downloads\ic_bootstrap\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Definição dos rótulos BIO
lista_rotulos = ["O", "B-WRONG", "I-WRONG"]
label_to_id = {label: i for i, label in enumerate(lista_rotulos)}
id_to_label = {i: label for i, label in enumerate(lista_rotulos)}

In [5]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove os índices de ignorados (-100) e converte para labels de texto
    true_predictions = [
        [id_to_label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id_to_label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    
    precision = results["overall_precision"]
    recall = results["overall_recall"]

    # F0.5 Score
    if precision + recall == 0: # Para evitar divisões por 0
        f0_5=0.0

    else:
        f0_5= (1+0.5**2) * (precision*recall) / ((0.5**2 * precision) + recall)

    return {
        "precision": precision,
        "recall": recall,
        "f0_5": f0_5,
        "accuracy": results["overall_accuracy"],
    }

In [6]:
# A versão PyTorch precisa ter suporte ao CUDA, então rode o pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126

In [7]:
import torch
device = torch.device('cuda')
print(f"Usando dispositivo: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")

Usando dispositivo: cuda
PyTorch version: 2.12.0+cu126
CUDA available: True
CUDA version: 12.6


In [8]:
# Lê o início do arquivo TSV
import torch
df = pd.read_csv('train_bio.tsv', sep='\t', encoding='latin-1', quoting=3)
print(df.head())


          SentenÃ§a Original NÂ°1
O                               O
maior                           O
mal                             O
da                              O
sociedade                       O


In [9]:
# Lê arquivo CONLL (TSV)

def ler_conll(
    path: str,
    *,
    ignore_header_regex: str = r"^\s*Sentença\s+Original\s+N[°º]\s*\d+",
    allow_space_in_token: bool = False,
    strict_two_columns: bool = False,
    verbose: bool = True,
) -> Tuple[List[List[str]], List[List[str]]]:
    header_pat = re.compile(ignore_header_regex, flags=re.IGNORECASE)
    tokens_per_sent, tags_per_sent = [], []
    cur_tokens, cur_tags = [], []

    def flush_sentence():
        nonlocal cur_tokens, cur_tags
        if cur_tokens:
            tokens_per_sent.append(cur_tokens)
            tags_per_sent.append(cur_tags)
        cur_tokens, cur_tags = [], []

    with open(path, "r", encoding="utf-8") as f:
        for lineno, raw in enumerate(f, start=1):
            line = raw.rstrip("\n")

            # Ignora linhas de cabeçalho por sentença
            if header_pat.search(line):
                flush_sentence()
                continue

            # Delimitador de sentença
            if line.strip() == "":
                flush_sentence()
                continue

            # Tenta split por TAB primeiro, senão por espaço
            if "\t" in line:
                parts = line.split("\t")
            else:
                # Divide por qualquer espaço
                parts = line.strip().split()

            if strict_two_columns:
                if len(parts) != 2:
                    if verbose:
                        print(f"[AVISO] Linha {lineno} ignorada (esperado 2 colunas): {line}")
                    continue
                token, tag = parts[0], parts[1]
            else:
                if len(parts) < 2:
                    if verbose:
                        print(f"[AVISO] Linha {lineno} ignorada (colunas insuficientes): {line}")
                    continue
                if allow_space_in_token:
                    # Última coluna é tag; o resto é token (pode conter espaços/tabs)
                    *token_parts, tag = parts
                    token = " ".join(token_parts)
                else:
                    # Primeira é token, última é tag (se houver colunas no meio, descartamos/avisamos)
                    token, tag = parts[0], parts[-1]
                    if len(parts) > 2 and verbose:
                        print(f"[OBSERVAÇÃO] Linha {lineno} possuía colunas extras; usando 1ª como token e última como tag.")

            # Sanidade básica da tag
            if not tag or tag.isspace():
                if verbose:
                    print(f"[AVISO] Linha {lineno} com tag vazia; ignorada: {line}")
                continue

            cur_tokens.append(token)
            cur_tags.append(tag)

    # Última sentença (se não terminar com linha em branco)
    flush_sentence()

    return tokens_per_sent, tags_per_sent

In [10]:
# Mapeia Label2ID e ID2Label

def build_label_maps(tags_per_sent: List[List[str]]) -> Tuple[Dict[str, int], Dict[int, str]]:
    labels = set()
    for sent_tags in tags_per_sent:
        labels.update(sent_tags)

    labels = list(labels)

    if "O" in labels:
        labels.remove("O")
        labels = ["O"] + sorted(labels)
    else:
        labels = sorted(labels)

    label2id = {lab: i for i, lab in enumerate(labels)} # Converte texto em número ['O': 0, 'B-WRONG': 1, 'I-WRONG': 2]
    id2label = {i: lab for lab, i in label2id.items()} # Converte número em texto [0: 'O', 1: 'B-WRONG', 2: 'I-WRONG']
    return label2id, id2label

In [11]:
# Constrói um dataset Hugging Face com colunas de tokens (List[str]) e ner_tags (List[int]).

def make_hf_dataset_from_tokens(
    tokens_per_sent: List[List[str]],
    tags_per_sent: List[List[str]],
    label2id: Optional[Dict[str, int]] = None,
) -> Tuple[Dataset, Dict[str, int]]:

    # Verifica se as sentenças de tokens e as sentenças de tags são equivalentes (mesmo número)
    assert len(tokens_per_sent) == len(tags_per_sent), "tokens e tags desalinhados"

    # Associa Label2ID ao mapeamento das tags feito anteriormente
    if label2id is None:
        label2id, _ = build_label_maps(tags_per_sent)

    # Tags desconhecidas serão associadas à tag O
    O_ID = label2id.get("O", 0)

    # Mapeamento com tratamento de erro
    encoded_tags = []
    for tags in tags_per_sent:
        sentence_tags = []
        for tag in tags:
            sentence_tags.append(label2id.get(tag, O_ID)) # Verifica se tag está mapeada ou se uso o ID 0
        encoded_tags.append(sentence_tags)

    ds = Dataset.from_dict({"tokens": tokens_per_sent, "ner_tags": encoded_tags})
    return ds, label2id

In [12]:
# Função para equiparar lista de labels com lista de tokens

def tokenize_and_align_labels(batch, tokenizer, label_all_subtokens: bool = False):
    tokenized = tokenizer(
        batch["tokens"],
        truncation=True,
        padding=True,
        is_split_into_words=True,
    )

    all_labels = []
    for i, word_labels in enumerate(batch["ner_tags"]):
        # Mapeia o subtoken com a palavra original
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                # Caractere especial é ignorado
                label_ids.append(-100)
            elif word_id != previous_word_id:
                # Primeiro pedaço da palavra recebe tag original
                label_ids.append(word_labels[word_id])
            else:
                # Sub-token da mesma palavra (word) é alinhado à label original
                label_ids.append(word_labels[word_id] if label_all_subtokens else -100)
            previous_word_id = word_id

        all_labels.append(label_ids)

    tokenized["labels"] = all_labels
    return tokenized


In [13]:
# Função que retorna o objeto DatasetDict para o treinamento do modelo a partir das funções anteriores.

def load_custom_conll_dataset(
    files: Dict[str, str] | str,
    *,
    fixed_label2id: Optional[Dict[str, int]] = None,
    allow_space_in_token: bool = False,
    strict_two_columns: bool = False,
    verbose: bool = True,
) -> Tuple[DatasetDict, Dict[str, int], Dict[int, str]]:

    if isinstance(files, str):
        files = {"train": files}

    datasets_splits = {}
    cache_tokens = {}
    cache_tags = {}

    # Lê os splits
    for split_name, path in files.items():
        tokens_per_sent, tags_per_sent = ler_conll(
            path,
            allow_space_in_token=allow_space_in_token,
            strict_two_columns=strict_two_columns,
            verbose=verbose,
        )
        cache_tokens[split_name] = tokens_per_sent
        cache_tags[split_name] = tags_per_sent

    # Determinar o mapeamento de labels
    if fixed_label2id is not None:
        # Mapeamento fixo (BIO)
        accumulated_label2id = fixed_label2id
    else:
        # Remapeaia todas as tags caso não tenha labels definidas (fallback)
        all_tags = []
        for split_name in files.keys():
            all_tags.extend(cache_tags[split_name])
        accumulated_label2id, _ = build_label_maps(all_tags)

    accumulated_id2label = {i: lab for lab, i in accumulated_label2id.items()}

    # Constrói datasets usando o mapeamento (BIO)
    for split_name in files.keys():
        ds, _ = make_hf_dataset_from_tokens(
            cache_tokens[split_name],
            cache_tags[split_name],
            label2id=accumulated_label2id,
        )
        datasets_splits[split_name] = ds

    dset = DatasetDict(datasets_splits)
    return dset, accumulated_label2id, accumulated_id2label

In [14]:
# Carregamento do Dataset já com os labels BIO
path_treino = "train_bio.tsv"
path_teste = "test_bio.tsv"
path_validacao = "val_bio.tsv"

dataset, _, _ = load_custom_conll_dataset(
      files={"train": path_treino, "test": path_teste, "validation": path_validacao},
      fixed_label2id=label_to_id,
      allow_space_in_token=False,
      strict_two_columns=False,
      verbose=True,
)

# Unificação dos dados para que o sorteio 80/20 seja global a cada rodada
full_dataset = concatenate_datasets([
    dataset["train"],
    dataset["test"],
    dataset["validation"]
])

num_experimentos = 30
resultados_experimentos = [] # Lista para armazenar métricas de cada rodada

print(full_dataset)
print("labels:", label_to_id)

Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 19250
})
labels: {'O': 0, 'B-WRONG': 1, 'I-WRONG': 2}


In [15]:
# 1. Verify id2label keys are integers
print("id2label key types:", [type(k) for k in id_to_label.keys()])

# 2. Verify label_to_id values are integers
print("label_to_id value types:", [type(v) for v in label_to_id.values()])

# 3. Check for consistency
print(f"Number of labels in list: {len(lista_rotulos)}")
print(f"Number of entries in id2label: {len(id_to_label)}")

id2label key types: [<class 'int'>, <class 'int'>, <class 'int'>]
label_to_id value types: [<class 'int'>, <class 'int'>, <class 'int'>]
Number of labels in list: 3
Number of entries in id2label: 3


In [16]:
# Testar pip install --upgrade torch torchvision torchaudio

In [17]:
tokenizer = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")

# Tokenização + alinhamento das labels
tokenized_dataset = dataset.map(
    lambda batch: tokenize_and_align_labels(batch, tokenizer, label_all_subtokens=False), # Aplica a função para alinhamento de tokens e tags
    batched=True # Processa várias sentenças simultaneamente
)

# Carregamento do Modelo
model = AutoModelForTokenClassification.from_pretrained(
    "neuralmind/bert-base-portuguese-cased",
    num_labels=len(lista_rotulos),
    id2label=id_to_label,
    label2id=label_to_id,
    use_safetensors=True
)

model.to(device)

print(f"\nModelo movido para: {device}")
print(f"\nOs rótulos foram definidos como: \n{model.config.id2label}")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 12608.58it/s]



Modelo movido para: cuda

Os rótulos foram definidos como: 
{0: 'O', 1: 'B-WRONG', 2: 'I-WRONG'}


In [18]:
# Configurações de Treinamento

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",       # Estratégia de save igual à de eval
    save_total_limit=1,
    load_best_model_at_end=True, # Salvamento e carregamento do melhor modelo ao final
    metric_for_best_model="f0_5",  # O critério é o F0.5 Score
    fp16=True,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    num_train_epochs=3,         # 3 épocas
    weight_decay=0.01,
    logging_dir="./logs",
    report_to=[],
)

In [19]:
# Cria o data collator que faz padding dinâmico
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
)

In [20]:
nome_modelo ="neuralmind/bert-base-portuguese-cased"
# Treinamento do modelo
import gc # Limpeza de memória antes do loop de experimentos

model = AutoModelForTokenClassification.from_pretrained(
    nome_modelo,
    num_labels=len(lista_rotulos),
    id2label=id_to_label,
    label2id=label_to_id
)

# 2. Argumentos de Treinamento
num_experimentos = 30
resultados_experimentos = []

print(f"** INÍCIO DO CICLO DE {num_experimentos} EXPERIMENTOS COM BERT BASE ***")

for i in range(num_experimentos):
    current_seed = 100 + i

    # Requisito 1: Shuffle Dinâmico em memória
    ds_split = full_dataset.shuffle(seed=current_seed).train_test_split(test_size=0.2, seed=current_seed)

    # Tokenização do split atual
    tokenized_ds = ds_split.map(
        lambda batch: tokenize_and_align_labels(batch, tokenizer, label_all_subtokens=False),
        batched=True,
        load_from_cache_file=False
    )

    # Requisito: Reinicialização do Modelo (Essencial para independência estatística)
    model = AutoModelForTokenClassification.from_pretrained(
        nome_modelo,
        num_labels=len(lista_rotulos),
        id2label=id_to_label,
        label2id=label_to_id
    ).to(device)

    output_dir_atual = f"./temp_results/exp_{i+1}"

    training_args = TrainingArguments(
        output_dir=output_dir_atual,
        eval_strategy="no",
        learning_rate=2e-5,
        per_device_train_batch_size=4,      # Batch size reduzido para evitar Out Of Memory Error
        gradient_accumulation_steps=4,        # Acumula gradientes para simular batch size de 16
        num_train_epochs=3,
        weight_decay=0.01,
        disable_tqdm=True,
        report_to=[],
        fp16=torch.cuda.is_available()
    )

    # Requisito 2: Trainer configurado dentro do loop
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds["train"],
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    # Execução do treino da rodada i
    trainer.train()

    # Requisito 3: Coleta de métricas essenciais
    metrics = trainer.evaluate(eval_dataset=tokenized_ds["test"])
    resultados_experimentos.append({
        'Precision': metrics['eval_precision'],
        'Recall': metrics['eval_recall'],
        'F0.5-Score': metrics['eval_f0_5'],
        'Accuracy': metrics['eval_accuracy']
    })

    print(f"✅ Rodada {i+1}/30 finalizada (Seed: {current_seed}) | F0_5: {metrics['eval_f0_5']:.4f}")

# Limpeza de memória após os experimentos
del model
del trainer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# --- Requisito 4: Agregação Estatística Final ---
df_resultados = pd.DataFrame(resultados_experimentos)
df_resultados.to_csv("resultados_bertbase.csv", index=False)
print("\n Os resultados dos 30 experimentos foram salvos em 'resultados_bertbase.csv'")


stats_media = df_resultados.mean()
stats_desvio = df_resultados.std()

print("\n" + "="*45)
print("📊 RELATÓRIO ESTATÍSTICO DE SPANS (n=30)")
print("="*45)
for metrica in df_resultados.columns:
    print(f"{metrica:10}: {stats_media[metrica]:.4f} (± {stats_desvio[metrica]:.4f})")
print("="*45)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 12611.08it/s]


** INÍCIO DO CICLO DE 30 EXPERIMENTOS COM BERT BASE ***


Loading weights: 100%|██████████| 197/197 [00:00<?, ?it/s]


{'loss': '1.05', 'grad_norm': '7.437', 'learning_rate': '1.655e-05', 'epoch': '0.5195'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.38it/s]


...

{'loss': '0.6206', 'grad_norm': '9.254', 'learning_rate': '6.161e-06', 'epoch': '2.077'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.82it/s]


{'loss': '0.5501', 'grad_norm': '7.609', 'learning_rate': '2.7e-06', 'epoch': '2.596'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.40it/s]


{'train_runtime': '6352', 'train_samples_per_second': '7.273', 'train_steps_per_second': '0.455', 'train_loss': '0.71', 'epoch': '3'}
{'eval_loss': '0.2195', 'eval_precision': '0.7041', 'eval_recall': '0.6366', 'eval_f0_5': '0.6895', 'eval_accuracy': '0.9454', 'eval_runtime': '96.68', 'eval_samples_per_second': '39.82', 'eval_steps_per_second': '4.985', 'epoch': '3'}
✅ Rodada 30/30 finalizada (Seed: 129) | F0_5: 0.6895

 Os resultados dos 30 experimentos foram salvos em 'resultados_bertbase.csv'

📊 RELATÓRIO ESTATÍSTICO DE SPANS (n=30)
Precision : 0.6917 (± 0.0088)
Recall    : 0.6199 (± 0.0103)
F0.5-Score: 0.6760 (± 0.0078)
Accuracy  : 0.9471 (± 0.0020)
